In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
train_df = pd.read_csv("/content/GSE98320.csv")
val_df   = pd.read_csv("/content/GSE129166.csv")

X_train = train_df.drop(columns=["sample_id", "diagnosis"])
y_train = train_df["diagnosis"]
X_val   = val_df.drop(columns=["sample_id", "diagnosis"])[X_train.columns]
y_val   = val_df["diagnosis"]

In [ ]:
class_labels = sorted(y_train.unique())

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=5000, random_state=42)),
])
param_grid = {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100]}

In [ ]:
def print_metrics(y_true, y_pred, label):
    print(f"\n=== RIDGE — {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=class_labels, zero_division=0)
    for cls, pi, ri, fi in zip(class_labels, p, r, f):
        print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
    print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")

In [ ]:
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
X_train_r, y_train_r = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
oof_pred = np.empty(len(y_train_r), dtype=object)

for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X_train_r, y_train_r)):
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    search = GridSearchCV(clone(pipeline), param_grid, cv=inner_cv, scoring="accuracy", n_jobs=-1)
    search.fit(X_train_r.iloc[tr_idx], y_train_r.iloc[tr_idx])
    oof_pred[te_idx] = search.predict(X_train_r.iloc[te_idx])
    print(f"  fold {fold+1}/10 done, best params={search.best_params_}")

print_metrics(y_train_r, oof_pred, "Cross-Validation Performance (GSE98320)")

  fold 1/10 done, best params={'clf__C': 0.01}
  fold 2/10 done, best params={'clf__C': 0.01}
  fold 3/10 done, best params={'clf__C': 0.01}
  fold 4/10 done, best params={'clf__C': 0.01}
  fold 5/10 done, best params={'clf__C': 0.01}
  fold 6/10 done, best params={'clf__C': 0.01}
  fold 7/10 done, best params={'clf__C': 0.01}
  fold 8/10 done, best params={'clf__C': 0.01}
  fold 9/10 done, best params={'clf__C': 0.01}
  fold 10/10 done, best params={'clf__C': 0.01}

=== RIDGE — Cross-Validation Performance (GSE98320) ===
Accuracy: 0.9102
  ABMR   | precision=0.8726  recall=0.8405  f1=0.8562
  NR     | precision=0.9258  recall=0.9509  f1=0.9382
  TCMR   | precision=0.9028  recall=0.8025  f1=0.8497
  MACRO  | precision=0.9004  recall=0.8646  f1=0.8814


In [ ]:
final_search = GridSearchCV(clone(pipeline), param_grid,
                             cv=StratifiedKFold(3, shuffle=True, random_state=42),
                             scoring="accuracy", n_jobs=-1)
final_search.fit(X_train, y_train)
val_pred = final_search.predict(X_val)
print_metrics(y_val, val_pred, "Independent Validation Performance (GSE129166)")


=== RIDGE — Independent Validation Performance (GSE129166) ===
Accuracy: 0.9481
  ABMR   | precision=0.7895  recall=1.0000  f1=0.8824
  NR     | precision=1.0000  recall=0.9333  f1=0.9655
  TCMR   | precision=1.0000  recall=1.0000  f1=1.0000
  MACRO  | precision=0.9298  recall=0.9778  f1=0.9493


In [ ]:
val_mask = y_val != "TCMR"
X_val_no_tcmr = X_val[val_mask]
y_val_no_tcmr = y_val[val_mask]
val_class_labels = sorted(y_val_no_tcmr.unique())

print(f"Excluded {(~val_mask).sum()} TCMR sample(s), evaluating on {len(y_val_no_tcmr)} remaining samples")

val_pred = final_search.predict(X_val_no_tcmr)

# Standalone printer for this cell only -- avoids touching the original
# print_metrics() which was hardcoded to the 3-class label set
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

print(f"\n=== Independent Validation Performance (GSE129166, TCMR excluded) ===")
print(f"Accuracy: {accuracy_score(y_val_no_tcmr, val_pred):.4f}")
p, r, f, _ = precision_recall_fscore_support(y_val_no_tcmr, val_pred, labels=val_class_labels, zero_division=0)
for cls, pi, ri, fi in zip(val_class_labels, p, r, f):
    print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")

Excluded 2 TCMR sample(s), evaluating on 75 remaining samples

=== Independent Validation Performance (GSE129166, TCMR excluded) ===
Accuracy: 0.9467
  ABMR   | precision=0.7895  recall=1.0000  f1=0.8824
  NR     | precision=1.0000  recall=0.9333  f1=0.9655
  MACRO  | precision=0.8947  recall=0.9667  f1=0.9239
